In [1]:
!pip install openai-agents -q

In [2]:
!uv --version
!uvx --version

uv 0.13.9 (6b2d39dcc 2026-11-17 aarch64-apple-darwin)
uvx 0.13.9 (6b2d39dcc 2026-11-17 aarch64-apple-darwin)


#### Benefits of MCP

1. The messy part gets absorbed by the server.
2. The LLM now sees one clear standard.
3. The MCP server is written once and everyone reuses it.
4. You can add tools to the LLM quickly.

#### Risks of MCP

1. A bad server runs on your machine with your full permissions.
2. "Official-looking" doesn't mean safe (many are community/solo-maintained).
3. Server content can carry hidden prompt-injection attacks.
4. Remote servers mean handing your data and access to a third party.
5. A trusted server can turn malicious in an update (rug pull).

#### MCP Types

HOST -> CLIENT
1. Client 1 -> local MCP server
2. Client 2 -> local MCP server -> API -> outside
3. Client 3 -> remote "hosted" MCP server (w Open Authentication Credentials)

Go to any MCP Server Directory and find the one you need

### Setup

In [3]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

MODEL = "gpt-4.1-mini"

In [4]:
WEB_AGENT_PROMPT = """
You are a helpful assistant.
When the user asks about a web page, if you can access the internet, 
then read that page first, then answer based on what you actually read.
Always cite the URL.
If you cannot access the internet, then say so.
"""

In [5]:
web_agent = Agent(
    name="Web-Agent",
    instructions=WEB_AGENT_PROMPT,
    model=MODEL,
)

In [6]:
result = await Runner.run(
    web_agent,
    input="Who is the guest mentioned on https://www.superdatascience.com/999",
    max_turns=10,
)

print(f"Last Agent: {result.last_agent.name}")
print(f"------")
display(Markdown(result.final_output))

Last Agent: Web-Agent
------


I currently do not have the capability to browse the internet in real-time. Therefore, I cannot access the page https://www.superdatascience.com/999 directly.

If you can provide the content or key details from the page, I'd be happy to help analyze or summarize the guest mentioned.

### Launch our first MCP server

#### Step 1: Describe launch parameters

In [17]:
fetch_server_params = {
    "command": "uvx",
    "args": ["mcp-server-fetch"],
}

#### Let's investigate the server

In [19]:
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, client_session_timeout_seconds=60) as server:
    tools = await server.list_tools()
    
    print(f"✅ Connected. The server offers {len(tools)} tool(s): \n")
    for tool in tools:
        print(f"🔧 {tool.name}")
        print(f"{tool.description}")

✅ Connected. The server offers 1 tool(s): 

🔧 fetch
Fetches a URL from the internet and optionally extracts its contents as markdown.

Although originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.


#### Step 2: Launch & Connect to the server

In [18]:
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, client_session_timeout_seconds=60) as server:
    web_agent = Agent(
        name="Web-Agent",
        instructions=WEB_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server]
    )

    result = await Runner.run(
        web_agent,
        input="Who is the guest mentioned on https://www.superdatascience.com/999",
        max_turns=10,
    )

    print(f"Last Agent: {result.last_agent.name}")
    print(f"------")
    display(Markdown(result.final_output))

Last Agent: Web-Agent
------


The guest mentioned on the SuperDataScience podcast episode 999 is Chip Huyen. She is known for her record-breaking book “AI Engineering,” the most-read title on the O’Reilly platform last year. Chip builds AI products, has worked with Snorkel AI and NVIDIA, founded an AI infrastructure startup, and has taught Machine Learning Systems Design at Stanford. The episode features a conversation between Chip Huyen and host Jon Krohn about the AI landscape, AI engineering versus machine learning engineering, the practical aspects of building AI systems, physical AI, robotics, and future career advice in AI.

Source: https://www.superdatascience.com/999